In [1]:
from functions import extract_esm2_embeddings_from_fasta, merge_and_cleanup_embeddings, build_X_y_from_embeddings, evaluate_mlp, evaluate_df, filter_interactions_by_two_scores,  evaluate_by_score_ranges
import pandas as pd

In [2]:
# path to interactions
interactions = "data/comparation_data/animal_0-20_0-20.csv"

# path to fasta
fasta = "data/comparation_data/animal_0-20_0-20.fasta"

# path to predictions
output_pred = 'predictions.csv'

# path to predictions splited in representation score
output_pred_score = 'predictions_by_representation_score.csv'

# path to esm2_t30_150M_UR50D.pt
esm_path = "esm2_t30_150M_UR50D.pt"

# path to alingment 
aling = pd.read_csv('data/aling/animal_test_aling.csv')

output_embbedins = fasta.replace(".fasta", "")+'_feats'

extract_esm2_embeddings_from_fasta(
    fasta_path=fasta,
    output_prefix=output_embbedins,
    batch_size=5,
    model_path=esm_path
)
print('')


merge_and_cleanup_embeddings(output_prefix=output_embbedins)
print('')


X, y, original_ppi = build_X_y_from_embeddings(
    feat_path=output_embbedins+'.csv',
    combined_path=interactions
)
print('')

y_pred, y_proba = evaluate_mlp(X, y)


df_save = original_ppi.copy()
df_save['y_pred'] = y_pred
df_save['y_proba'] = y_proba

df_save.to_csv(output_pred)
print('Predictions saved in:', output_pred)

Shape X: (288, 1280)
Shape y: (288,)


=== Final Metrics ===
Accuracy:    0.6736
Precision:   0.7016
Recall:      0.6042
Specificity: 0.7431
F1-score:    0.6493
AUPR:        0.7735
AUROC:       0.7647
Predictions saved in: predictions.csv


In [7]:
faixas = [
    ["80-100", "80-100"],
    ["80-100", "60-80"],
    ["80-100", "40-60"],
    ["80-100", "20-40"],
    ["80-100", "0-20"],

    ["60-80", "60-80"],
    ["60-80", "40-60"],
    ["60-80", "20-40"],
    ["60-80", "0-20"],

    ["40-60", "40-60"],
    ["40-60", "20-40"],
    ["40-60", "0-20"],
    
    ["20-40", "20-40"],
    ["20-40", "0-20"],
    
    ["0-20", "0-20"]
]

df_all_results, df_all_predictions = evaluate_by_score_ranges(
    ranges=faixas,
    alignment_df=aling,
    original_ppi_df=original_ppi,
    y=y,
    y_pred=y_pred,
    y_proba=y_proba
)

print(df_all_results)
df_all_predictions.to_csv(output_pred_score)

  Range1 Range2  Threshold  Precision    Recall        F1  Specificity  \
0   0-20   0-20        0.5   0.701613  0.604167  0.649254     0.743056   

   Standard_Accuracy      AUPR    AUROC  
0           0.673611  0.773484  0.76466  
